## GC Steps
Consider a `Person` class which gets initialised to create a new object on heap.

In [ ]:
class Person {
    String name;
    Person(String name) { this.name = name; }
    void walk() { System.out.println(name + " is walking"); }
}

As mentioned, objects allocated using the constructor are stored in heap memory.  
![Heap](https://i.imgur.com/gSCevSI.png)

Over time some objects become unreferenced. These objects are candidates for GC and can be removed to reclaim memory. 

The first step in GC is *marking*, the garbage collector traverses from *GC roots* and marks all reachable objects. Objects that are not marked are candidates for removal.  
![Marking](https://i.imgur.com/KXyEIMB.png)  

GC roots are the starting points to do reachability analysis. These are:
- Local variables on thread stacks
- Active thread objects
- Static fields of loaded classes
- JNI references

Any object reachable (directly or transitively) from a GC root is considered live.

An object can become eligible for garbage collection in many different ways:
1. Nullifying reference variable:

In [ ]:
Person p1 = new Person("John Doe");
p1 = null; // the above created object is now eligible for GC

2. Reassigning:

In [ ]:
Person p1 = new Person("John Doe");
Person p2 = new Person("Jane Doe");
p1 = p2; // John Doe available for GC

3. Object created inside a method:

In [ ]:
public void someMethod(){
    Person p1 = new Person("John Doe");
}
// p1 now eligible for GC

4. Anonymous object:

In [ ]:
new Person("Steve Jobs").walk();
// Steve Jobs available for GC

5. Objects with just internal (circular) references:

<img src="https://miro.medium.com/max/875/0*XY9nzQPR3LXWWmZj" width="500" height="auto">

The next step is *deleting*. After deletion, we get lots of empty space.  
![Deletion](https://i.imgur.com/AJxJFmG.png)

The final step is *compacting* : referenced objects are moved together to eliminate fragmentation.  
![Compacting](https://i.imgur.com/TqoE9yo.png)

## Generational GC
In a typical program, most of the objects are short lived (the *weak generational hypothesis*). Therefore it makes sense to divide the heap area into different parts depending upon the age of objects.  
![GC Generations](https://i.imgur.com/r3fDW8P.png)

**Young Generation:** Newly created objects are put in *Eden* space in the Young Generation. When Eden space fills up, it causes a *minor garbage collection* to trigger intended to cleanup young generation. The young generation also contains survivor spaces *S0* and *S1*. 

A minor garbage collection involves the steps listed in the diagram below. This process is called `mark-copy`:  
![Young generation GC process](https://i.imgur.com/7fqKOis.png)  
Note that in the above process the copying is done in a way that compaction is not required. We can afford to copy objects here since we don't expect a lot of live objects in young generation. At any given point in time either S0 is occupied and S1 is empty or vice-versa. After copying, the eden space and one survivor space are simply wiped out. 


**Old Generation:** After a minor GC, when aged objects reach a certain age threshold they are promoted from young generation to old generation. Garbage collection involving old generation heap is called *major garbage collection*. This event is triggered when old generation is approaching being full or a minor GC wants to promote to old generation which has not enough space.

Major garbage collection in old generation utilises `mark-sweep-compact` algorithm which was explained earlier on this page: find live objects → clear dead objects → pack live ones. Compaction is the expensive part which is why major GC is slower than minor GC.

*Full GC* involves both old and young generation heap.

**Metaspace:** In addition to the object heap, the JVM maintains *metaspace* (which replaced *PermGen* in Java 8) to store class metadata, method bytecodes, and constant pools. Metaspace is allocated in native memory and grows automatically, but exhaustion can still trigger GC and cause `OutOfMemoryError`.

In traditional collectors (`Serial`, `Parallel`), both minor and major GC are stop-the-world events which means that all application threads are paused while the garbage collection thread runs. Modern collectors like `G1`, `ZGC`, and `Shenandoah` reduce or eliminate stop-the-world pauses through concurrent processing.

## GC Implementations
When considering a certain GC implementation we look at the below two factors:
- **Throughput:** percentage of total time not spent in garbage collection considered over long periods of time. So if the application spends 10% of its time in garbage collection the throughput is 90%. A good GC implementation would maximise throughput.  
- **Latency:** GC operation pauses code execution. Latency measures pauses induced by the JVM as it performs garbage collection.

### Serial Garbage Collector
This is the simplest GC implementation. It uses `mark-copy` for the Young Generation and `mark-sweep-compact` for the Old Generation.  
![Serial GC](./images/serial_gc.png)  

To use this:
```sh
$ java -XX:+UseSerialGC com.mypackages.MyExecutableClass
```
Recommended only for the JVM with a couple of hundreds megabytes heap size, running in an environment with a single CPU.

### Parallel Garbage Collector
Similar to Serial GC, it uses `mark-copy` in the Young Generation and `mark-sweep-compact` in the Old Generation. Both Young and Old collection run *marking and copying/compacting* phases using multiple threads, hence the name 'Parallel'.  
![Serial GC](./images/parallel_gc.png)  

To use this:
```sh
$ java -XX:+UseParallelGC com.mypackages.MyExecutableClass
```

This collector results in high throughput (overall the time spent in GC is less) and is suitable for multi-core machines. However, since all phases of the collection happen without any interruptions, it is still susceptible to long pauses during which your application threads are stopped. Thus, it is not suitable for applications where latency is a concern. In short, it has:
- high throughput since the total time lost doing GC is small
- high latency since we still have long single stalls

### G1 Garbage Collector
G1GC instead of using one big young and old generation, it divides the heap into many equal-sized regions. Each region can serve as eden, survivor, or old space, allowing G1 to flexibly resize generations.  
![G1GC](./images/g1gc.png)

The goal of the G1 collector is to achieve a predictable soft-target pause time, defined through `-XX:MaxGCPauseMillis`, while also maintaining consistent application throughput. We can request stop-the-world pauses to be no longer than *x* milliseconds within any given *y*-millisecond time range (e.g., no more than 5 milliseconds in any given second). G1 will do its best to meet this goal with high probability.

Here is how part of the process works:
![Young Collection](./images/g1gc_young_collection.png)

To use with a pause target:
```
java -XX:+UseG1GC -XX:MaxGCPauseMillis=200 com.mypackages.MyExecutableClass
```

### ZGC
ZGC is a concurrent, low-latency garbage collector designed for applications requiring sub-millisecond pause times, regardless of heap size. It can handle heaps from a few hundred megabytes to multi-terabyte sizes.

Key features:
- Pause times are typically under 1 ms and do not increase with heap or live-set size.
- Almost all GC work (marking, relocation, reference processing) is done concurrently.
- Uses colored pointers and load barriers.

To use:
```
java -XX:+UseZGC com.mypackages.MyExecutableClass
```

### Shenandoah GC
Shenandoah is another **low-pause concurrent** garbage collector (available in OpenJDK builds). Like ZGC, it aims to keep GC pause times short and independent of heap size by performing evacuation (compaction) concurrently with the application.

To use:
```
java -XX:+UseShenandoahGC com.mypackages.MyExecutableClass
```